In [ ]:
import requests
import os

print("="*80)
print("STEP 2a: AUTOMATED DATA INGESTION FROM SOURCE")
print("="*80)

# Source URL from St. Louis Metropolitan Police Department
crime_data_url = "https://slmpd.org/wp-content/uploads/2026/03/February2026.csv"

print(f"\n📡 Data Source: {crime_data_url}")
print("📥 Initiating automated download...")

try:
    # Automated HTTP GET request with timeout handling
    response = requests.get(crime_data_url, timeout=30)
    response.raise_for_status()  # Raises HTTPError for bad status codes (4xx, 5xx)
    
    # Define output path in workspace
    output_path = '/Workspace/final project/February2026.csv'
    
    # Write binary content to file
    with open(output_path, 'wb') as f:
        f.write(response.content)
    
    # Success logging with metrics
    print(f"\n✅ DOWNLOAD SUCCESSFUL")
    print(f"   • File Size: {len(response.content):,} bytes ({len(response.content)/1024/1024:.2f} MB)")
    print(f"   • Saved To: {output_path}")
    print(f"   • HTTP Status: {response.status_code}")
    print(f"   • Records Ready: File available for processing")
    
    # File verification
    if os.path.exists(output_path):
        print(f"   • Verification: ✓ File integrity confirmed")
    
except requests.exceptions.Timeout:
    print("\n❌ ERROR: Download timed out after 30 seconds")
    print("   • Suggestion: Check network connectivity or increase timeout")
except requests.exceptions.HTTPError as e:
    print(f"\n❌ ERROR: HTTP request failed - {e}")
    print("   • Suggestion: Verify URL is still valid")
except Exception as e:
    print(f"\n❌ ERROR: Unexpected error - {e}")

print("="*80)

In [ ]:
import numpy as np
import pandas as pd

print("=" * 80)
print("VALIDATION: STEP 3 - GEOGRAPHIC DATA INTEGRATION")
print("=" * 80)

validation_passed = True
issues = []

# ──────────────────────────────────────────────
# 1. INPUT DATAFRAME CHECKS
# ──────────────────────────────────────────────
print("\n[1] Checking input DataFrames...")

# Check df_crime exists and is non-empty
assert 'df_crime' in dir(), "df_crime is not defined"
assert len(df_crime) > 0, "df_crime is empty"
print(f"  ✓ df_crime loaded: {len(df_crime):,} rows, {df_crime.shape[1]} columns")

# Check df_zip exists and is non-empty
assert 'df_zip' in dir(), "df_zip is not defined"
assert len(df_zip) > 0, "df_zip is empty"
print(f"  ✓ df_zip loaded: {len(df_zip):,} rows")

# Check required columns in df_crime
required_crime_cols = ['Latitude', 'Longitude', 'IncidentDate', 'Neighborhood', 'CrimeAgainst']
missing_crime_cols = [c for c in required_crime_cols if c not in df_crime.columns]
if missing_crime_cols:
    issues.append(f"df_crime missing columns: {missing_crime_cols}")
    print(f"  ✗ df_crime missing columns: {missing_crime_cols}")
else:
    print(f"  ✓ df_crime has all required columns")

# Check required columns in df_zip
required_zip_cols = ['latitude', 'longitude', 'zip_code']
missing_zip_cols = [c for c in required_zip_cols if c not in df_zip.columns]
if missing_zip_cols:
    issues.append(f"df_zip missing columns: {missing_zip_cols}")
    print(f"  ✗ df_zip missing columns: {missing_zip_cols}")
else:
    print(f"  ✓ df_zip has all required columns")

# ──────────────────────────────────────────────
# 2. COORDINATE VALIDITY CHECKS (PRE-MAPPING)
# ──────────────────────────────────────────────
print("\n[2] Validating coordinates in df_crime...")

# Null coordinates
null_lat = df_crime['Latitude'].isna().sum()
null_lon = df_crime['Longitude'].isna().sum()
if null_lat > 0 or null_lon > 0:
    issues.append(f"Null coordinates: {null_lat} Latitude, {null_lon} Longitude")
    print(f"  ✗ Null Latitude: {null_lat:,} | Null Longitude: {null_lon:,}")
else:
    print(f"  ✓ No null coordinates")

# Out-of-range coordinates (basic global bounds)
invalid_lat = df_crime[(df_crime['Latitude'] < -90) | (df_crime['Latitude'] > 90)]
invalid_lon = df_crime[(df_crime['Longitude'] < -180) | (df_crime['Longitude'] > 180)]
if len(invalid_lat) > 0:
    issues.append(f"{len(invalid_lat)} rows have Latitude out of [-90, 90]")
    print(f"  ✗ Invalid Latitude values: {len(invalid_lat):,} rows")
else:
    print(f"  ✓ All Latitude values in valid range [-90, 90]")

if len(invalid_lon) > 0:
    issues.append(f"{len(invalid_lon)} rows have Longitude out of [-180, 180]")
    print(f"  ✗ Invalid Longitude values: {len(invalid_lon):,} rows")
else:
    print(f"  ✓ All Longitude values in valid range [-180, 180]")

# Zero-island coordinates (0, 0) often indicate missing data encoded as zero
zero_coords = df_crime[(df_crime['Latitude'] == 0) & (df_crime['Longitude'] == 0)]
if len(zero_coords) > 0:
    issues.append(f"{len(zero_coords)} rows have (0, 0) coordinates — likely missing data")
    print(f"  ⚠ Rows with (0.0, 0.0) coordinates: {len(zero_coords):,} — check for missing data encoded as zero")
else:
    print(f"  ✓ No (0, 0) placeholder coordinates")

# ──────────────────────────────────────────────
# 3. ZIP CODE REFERENCE DATA CHECKS
# ──────────────────────────────────────────────
print("\n[3] Validating df_zip reference data...")

null_zip_lat = df_zip['latitude'].isna().sum()
null_zip_lon = df_zip['longitude'].isna().sum()
null_zip_code = df_zip['zip_code'].isna().sum()
dup_zip = df_zip['zip_code'].duplicated().sum()

if null_zip_lat > 0 or null_zip_lon > 0 or null_zip_code > 0:
    issues.append(f"df_zip has nulls — lat: {null_zip_lat}, lon: {null_zip_lon}, zip_code: {null_zip_code}")
    print(f"  ✗ Nulls in df_zip — latitude: {null_zip_lat}, longitude: {null_zip_lon}, zip_code: {null_zip_code}")
else:
    print(f"  ✓ No nulls in df_zip reference columns")

if dup_zip > 0:
    print(f"  ⚠ Duplicate zip_code entries in df_zip: {dup_zip} — nearest-neighbor may be ambiguous")
else:
    print(f"  ✓ No duplicate zip codes in df_zip")

# ──────────────────────────────────────────────
# 4. POST-MAPPING CHECKS
# ──────────────────────────────────────────────
print("\n[4] Validating zip_code column after mapping...")

# Check zip_code column exists
if 'zip_code' not in df_crime.columns:
    issues.append("zip_code column not found in df_crime after mapping")
    print("  ✗ zip_code column missing — mapping may not have run")
    validation_passed = False
else:
    print("  ✓ zip_code column exists")

    # Null zip codes post-mapping
    null_zips = df_crime['zip_code'].isna().sum()
    if null_zips > 0:
        issues.append(f"{null_zips:,} rows have null zip_code after mapping")
        print(f"  ✗ Null zip_codes after mapping: {null_zips:,}")
    else:
        print(f"  ✓ No null zip_codes after mapping")

    # Coverage: every row should have a zip code
    total = len(df_crime)
    mapped = df_crime['zip_code'].notna().sum()
    coverage_pct = (mapped / total) * 100
    print(f"  ✓ Mapping coverage: {mapped:,} / {total:,} rows ({coverage_pct:.2f}%)")

    # Zip codes assigned must all exist in df_zip reference
    valid_zips = set(df_zip['zip_code'].dropna().astype(str))
    assigned_zips = set(df_crime['zip_code'].dropna().astype(str))
    rogue_zips = assigned_zips - valid_zips
    if rogue_zips:
        issues.append(f"Zip codes in df_crime not found in df_zip reference: {rogue_zips}")
        print(f"  ✗ Unrecognised zip codes assigned (not in df_zip): {rogue_zips}")
    else:
        print(f"  ✓ All assigned zip codes exist in df_zip reference")

    # Suspicious single-zip dominance (>80% of all records)
    top_zip, top_count = df_crime['zip_code'].value_counts().iloc[0], df_crime['zip_code'].value_counts().iloc[0]
    top_zip_label = df_crime['zip_code'].value_counts().index[0]
    top_pct = (top_count / total) * 100
    if top_pct > 80:
        issues.append(f"Zip code '{top_zip_label}' assigned to {top_pct:.1f}% of records — possible clustering issue")
        print(f"  ⚠ '{top_zip_label}' covers {top_pct:.1f}% of records — check for coordinate clustering")
    else:
        print(f"  ✓ No single zip code dominates (top: '{top_zip_label}' at {top_pct:.1f}%)")

# ──────────────────────────────────────────────
# 5. SUMMARY
# ──────────────────────────────────────────────
print("\n" + "=" * 80)
if not issues:
    print("✅  ALL VALIDATION CHECKS PASSED — Data integration looks clean.")
else:
    print(f"⚠️  VALIDATION COMPLETE WITH {len(issues)} ISSUE(S):")
    for i, issue in enumerate(issues, 1):
        print(f"  {i}. {issue}")
print("=" * 80)

In [ ]:
required_columns = [
    'zip_code', 'traffic_flow_index', 'road_quality_score',
    'avg_slope_percent', 'max_slope_percent',
    'overall_weather_risk', 'drivability_score'
]

missing_cols = [col for col in required_columns if col not in df_road_conditions.columns]

assert len(missing_cols) == 0, f"Missing columns: {missing_cols}"
print("✓ All required columns present")

In [ ]:
# Missing values
null_counts = df_road_conditions.isnull().sum()
assert null_counts.sum() == 0, f"Missing values found:\n{null_counts}"

# Duplicate zip codes
assert df_road_conditions['zip_code'].is_unique, "Duplicate zip codes found"

print("✓ No missing or duplicate data")